# 深度学习 HW02
## 多层感知机、正则化、数值稳定性、泛化与偏移

**Date:** 2026-05-28  
**Python:** 3.10

---
# Part 1: 多层感知机

## 1.1 非线性激活函数的重要性 (理论推导)

### 问题
单隐藏层MLP，隐藏层无激活（线性）：
$$h = W_1 x + b_1$$
$$o = W_2 h + b_2$$

证明该网络等价于单层神经网络。

### 推导
$$o = W_2(W_1 x + b_1) + b_2 = W_2 W_1 x + W_2 b_1 + b_2$$

令 $W' = W_2 W_1$，$b' = W_2 b_1 + b_2$，则：
$$o = W' x + b'$$

**结论：** 多层线性变换的复合仍为线性变换，无法增加模型表达能力。因此非线性激活函数至关重要。

---

## 1.2 激活函数导数推导 (理论)

### Sigmoid
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

### tanh
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

$$\tanh'(x) = 1 - \tanh^2(x)$$

## 1.3 编程：从零实现单隐藏层MLP（Fashion-MNIST多分类）

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 数据加载
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

In [ ]:
# 纯NumPy MLP 实现
class SimpleMLP:
    def __init__(self, input_size=784, hidden_size=128, output_size=10, lr=0.01):
        # Xavier初始化
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(1.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(1.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        self.lr = lr

    def relu(self, x):
        return np.maximum(0, x)

    def relu_grad(self, x):
        return (x > 0).astype(float)

    def softmax(self, x):
        e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return e_x / np.sum(e_x, axis=1, keepdims=True)

    def forward(self, X):
        self.X = X
        self.h = self.relu(X @ self.W1 + self.b1)
        self.o = self.h @ self.W2 + self.b2
        self.probs = self.softmax(self.o)
        return self.probs

    def cross_entropy_loss(self, y):
        m = y.shape[0]
        log_probs = -np.log(self.probs[np.arange(m), y] + 1e-8)
        return np.mean(log_probs)

    def backward(self, y):
        m = y.shape[0]
        dL_do = self.probs.copy()
        dL_do[np.arange(m), y] -= 1
        dL_do /= m

        dL_dW2 = self.h.T @ dL_do
        dL_db2 = np.sum(dL_do, axis=0, keepdims=True)

        dL_dh = dL_do @ self.W2.T
        dL_dh *= self.relu_grad(self.X @ self.W1 + self.b1)

        dL_dW1 = self.X.T @ dL_dh
        dL_db1 = np.sum(dL_dh, axis=0, keepdims=True)

        return dL_dW1, dL_db1, dL_dW2, dL_db2

    def update(self, dL_dW1, dL_db1, dL_dW2, dL_db2):
        self.W1 -= self.lr * dL_dW1
        self.b1 -= self.lr * dL_db1
        self.W2 -= self.lr * dL_dW2
        self.b2 -= self.lr * dL_db2

    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)

print("SimpleMLP 定义完成")

In [ ]:
# 训练MLP
mlp = SimpleMLP(input_size=784, hidden_size=128, output_size=10, lr=0.1)
epochs = 10
train_losses = []
train_accs = []

for epoch in range(epochs):
    epoch_loss = 0
    epoch_acc = 0
    n_batches = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.numpy().reshape(X_batch.shape[0], -1)
        y_batch = y_batch.numpy()
        
        # Forward
        mlp.forward(X_batch)
        loss = mlp.cross_entropy_loss(y_batch)
        epoch_loss += loss
        
        # Accuracy
        pred = np.argmax(mlp.probs, axis=1)
        epoch_acc += np.mean(pred == y_batch)
        
        # Backward & Update
        dL_dW1, dL_db1, dL_dW2, dL_db2 = mlp.backward(y_batch)
        mlp.update(dL_dW1, dL_db1, dL_dW2, dL_db2)
        
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    avg_acc = epoch_acc / n_batches
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")

print("\n训练完成")

In [ ]:
# 测试集评估
test_acc = 0
n_test = 0
for X_batch, y_batch in test_loader:
    X_batch = X_batch.numpy().reshape(X_batch.shape[0], -1)
    y_batch = y_batch.numpy()
    pred = mlp.predict(X_batch)
    test_acc += np.sum(pred == y_batch)
    n_test += len(y_batch)

test_acc /= n_test
print(f"Test Accuracy: {test_acc:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.subplot(1, 2, 2)
plt.plot(train_accs)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy')
plt.tight_layout()
plt.savefig('mlp_training.png', dpi=100)
plt.show()

---
# Part 2: 模型选择、权重衰减和丢弃法

## 2.1 过拟合与欠拟合 (理论)

### 定义
- **训练误差 (Training Error)：** 模型在训练集上的预测误差
- **泛化误差 (Generalization Error)：** 模型在未见过的测试数据上的预测误差

### 过拟合状态
当训练误差极低但泛化误差很高时，模型处于**过拟合**状态。

### 缓解方法
1. 减少模型复杂度（隐藏层大小、层数）
2. 增加训练数据量
3. 正则化（L1/L2权重衰减）
4. Dropout
5. 早停法 (Early Stopping)

---

## 2.2 K折交叉验证 (理论)

### 算法步骤
1. 将数据集均匀分为K份
2. For i = 1 to K:
   - 第i份作为验证集
   - 其余K-1份作为训练集
   - 训练模型并记录验证集性能
3. 取K次验证的平均性能作为最终评估

### 优点
- 充分利用数据
- 减少随机性影响
- 适合小数据集

## 2.3 编程：L2正则化 + Dropout实现

In [ ]:
class MLPWithRegularization:
    def __init__(self, input_size=784, hidden_size=128, output_size=10, lr=0.01, l2_lambda=0.001, dropout_rate=0.5):
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(1.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(1.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        self.lr = lr
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate
        self.is_training = True

    def relu(self, x):
        return np.maximum(0, x)

    def relu_grad(self, x):
        return (x > 0).astype(float)

    def softmax(self, x):
        e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return e_x / np.sum(e_x, axis=1, keepdims=True)

    def dropout_layer(self, X, is_training=True):
        """Dropout实现：训练时随机丢弃，测试时不丢弃"""
        if not is_training:
            return X
        mask = np.random.binomial(1, 1 - self.dropout_rate, X.shape) / (1 - self.dropout_rate)
        return X * mask

    def forward(self, X, is_training=True):
        self.X = X
        self.h_pre = X @ self.W1 + self.b1
        self.h = self.relu(self.h_pre)
        self.h_dropped = self.dropout_layer(self.h, is_training)
        self.o = self.h_dropped @ self.W2 + self.b2
        self.probs = self.softmax(self.o)
        return self.probs

    def cross_entropy_loss(self, y):
        m = y.shape[0]
        log_probs = -np.log(self.probs[np.arange(m), y] + 1e-8)
        ce_loss = np.mean(log_probs)
        # L2正则化项
        l2_loss = 0.5 * self.l2_lambda * (np.sum(self.W1**2) + np.sum(self.W2**2))
        return ce_loss + l2_loss

    def backward(self, y):
        m = y.shape[0]
        dL_do = self.probs.copy()
        dL_do[np.arange(m), y] -= 1
        dL_do /= m

        dL_dW2 = self.h_dropped.T @ dL_do + self.l2_lambda * self.W2
        dL_db2 = np.sum(dL_do, axis=0, keepdims=True)

        dL_dh = dL_do @ self.W2.T
        dL_dh *= self.relu_grad(self.h_pre)

        dL_dW1 = self.X.T @ dL_dh + self.l2_lambda * self.W1
        dL_db1 = np.sum(dL_dh, axis=0, keepdims=True)

        return dL_dW1, dL_db1, dL_dW2, dL_db2

    def update(self, dL_dW1, dL_db1, dL_dW2, dL_db2):
        # 权重衰减：W = W * (1 - η*λ) - η*∇L
        decay = 1 - self.lr * self.l2_lambda
        self.W1 = decay * self.W1 - self.lr * dL_dW1
        self.W2 = decay * self.W2 - self.lr * dL_dW2
        self.b1 -= self.lr * dL_db1
        self.b2 -= self.lr * dL_db2

    def predict(self, X):
        self.forward(X, is_training=False)
        return np.argmax(self.probs, axis=1)

print("MLPWithRegularization 定义完成")

In [ ]:
# 对比实验：无正则化 vs L2 vs Dropout
def train_and_evaluate(model, train_loader, val_loader, epochs=10):
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    for epoch in range(epochs):
        # 训练
        train_loss, train_acc = 0, 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.numpy().reshape(X_batch.shape[0], -1)
            y_batch = y_batch.numpy()
            model.forward(X_batch, is_training=True)
            loss = model.cross_entropy_loss(y_batch)
            train_loss += loss
            pred = np.argmax(model.probs, axis=1)
            train_acc += np.mean(pred == y_batch)
            dL_dW1, dL_db1, dL_dW2, dL_db2 = model.backward(y_batch)
            model.update(dL_dW1, dL_db1, dL_dW2, dL_db2)
        
        train_losses.append(train_loss / len(train_loader))
        train_accs.append(train_acc / len(train_loader))
        
        # 验证
        val_loss, val_acc = 0, 0
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.numpy().reshape(X_batch.shape[0], -1)
            y_batch = y_batch.numpy()
            model.forward(X_batch, is_training=False)
            loss = model.cross_entropy_loss(y_batch)
            val_loss += loss
            pred = np.argmax(model.probs, axis=1)
            val_acc += np.mean(pred == y_batch)
        
        val_losses.append(val_loss / len(val_loader))
        val_accs.append(val_acc / len(val_loader))
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}")
    
    return train_losses, val_losses, train_accs, val_accs

# 创建验证集
val_size = int(0.1 * len(train_data))
train_size = len(train_data) - val_size
train_subset, val_subset = torch.utils.data.random_split(train_data, [train_size, val_size])
train_loader_small = DataLoader(train_subset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=128, shuffle=False)

print(f"Train: {train_size}, Val: {val_size}")

In [ ]:
# 模型1：无正则化
print("\n=== 模型1: 无正则化 ===")
model1 = MLPWithRegularization(input_size=784, hidden_size=128, output_size=10, lr=0.1, l2_lambda=0.0, dropout_rate=0.0)
train_losses_1, val_losses_1, train_accs_1, val_accs_1 = train_and_evaluate(model1, train_loader_small, val_loader, epochs=10)

# 模型2：L2正则化
print("\n=== 模型2: L2正则化 ===")
model2 = MLPWithRegularization(input_size=784, hidden_size=128, output_size=10, lr=0.1, l2_lambda=0.001, dropout_rate=0.0)
train_losses_2, val_losses_2, train_accs_2, val_accs_2 = train_and_evaluate(model2, train_loader_small, val_loader, epochs=10)

# 模型3：Dropout
print("\n=== 模型3: Dropout ===")
model3 = MLPWithRegularization(input_size=784, hidden_size=128, output_size=10, lr=0.1, l2_lambda=0.0, dropout_rate=0.5)
train_losses_3, val_losses_3, train_accs_3, val_accs_3 = train_and_evaluate(model3, train_loader_small, val_loader, epochs=10)

In [ ]:
# 绘制对比曲线
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss曲线
ax = axes[0, 0]
ax.plot(train_losses_1, label='No Reg (Train)', linestyle='-', linewidth=2)
ax.plot(val_losses_1, label='No Reg (Val)', linestyle='--', linewidth=2)
ax.plot(train_losses_2, label='L2 (Train)', linestyle='-', linewidth=2)
ax.plot(val_losses_2, label='L2 (Val)', linestyle='--', linewidth=2)
ax.plot(train_losses_3, label='Dropout (Train)', linestyle='-', linewidth=2)
ax.plot(val_losses_3, label='Dropout (Val)', linestyle='--', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training vs Validation Loss')
ax.legend()
ax.grid()

# Accuracy曲线
ax = axes[0, 1]
ax.plot(train_accs_1, label='No Reg (Train)', linestyle='-', linewidth=2)
ax.plot(val_accs_1, label='No Reg (Val)', linestyle='--', linewidth=2)
ax.plot(train_accs_2, label='L2 (Train)', linestyle='-', linewidth=2)
ax.plot(val_accs_2, label='L2 (Val)', linestyle='--', linewidth=2)
ax.plot(train_accs_3, label='Dropout (Train)', linestyle='-', linewidth=2)
ax.plot(val_accs_3, label='Dropout (Val)', linestyle='--', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Training vs Validation Accuracy')
ax.legend()
ax.grid()

# 过拟合间隙：L2
ax = axes[1, 0]
ax.plot([l - t for l, t in zip(val_losses_2, train_losses_2)], marker='o', label='L2', linewidth=2)
ax.plot([l - t for l, t in zip(val_losses_1, train_losses_1)], marker='s', label='No Reg', linewidth=2)
ax.plot([l - t for l, t in zip(val_losses_3, train_losses_3)], marker='^', label='Dropout', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Generalization Gap (Val Loss - Train Loss)')
ax.set_title('Generalization Gap Comparison')
ax.legend()
ax.grid()
ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)

# 总结
ax = axes[1, 1]
ax.axis('off')
summary_text = f"""实验总结:

无正则化:
  最终验证Loss: {val_losses_1[-1]:.4f}
  最终验证Acc: {val_accs_1[-1]:.4f}
  过拟合间隙: {val_losses_1[-1] - train_losses_1[-1]:.4f}

L2正则化 (λ=0.001):
  最终验证Loss: {val_losses_2[-1]:.4f}
  最终验证Acc: {val_accs_2[-1]:.4f}
  过拟合间隙: {val_losses_2[-1] - train_losses_2[-1]:.4f}

Dropout (p=0.5):
  最终验证Loss: {val_losses_3[-1]:.4f}
  最终验证Acc: {val_accs_3[-1]:.4f}
  过拟合间隙: {val_losses_3[-1] - train_losses_3[-1]:.4f}
"""
ax.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('regularization_comparison.png', dpi=100)
plt.show()

print("\n对比实验完成")

---
# Part 3: 数值稳定性和激活函数

## 3.1 梯度消失与爆炸 (理论)

### 梯度计算链式法则
对于d层深网络，梯度包含乘积项：
$$\frac{\partial L}{\partial W_t} \propto \prod_{i=t}^{d-1} \frac{\partial h_{i+1}}{\partial h_i}$$

### 梯度爆炸
- 当 $|\frac{\partial h_{i+1}}{\partial h_i}| > 1$ 时，梯度逐层放大
- 原因：权重矩阵特征值 > 1，激活函数导数值大
- 表现：NaN、Loss爆炸

### 梯度消失
- 当 $|\frac{\partial h_{i+1}}{\partial h_i}| < 1$ 时，梯度逐层衰减
- 原因：Sigmoid导数 $\sigma'(x) \in (0, 0.25)$，权重矩阵特征值 < 1
- 表现：深层参数无法更新

### ReLU的优势
$$\text{ReLU}'(x) = \begin{cases} 1 & x > 0 \\ 0 & x < 0 \end{cases}$$

对于 $x > 0$ 的路径，导数为1，不会衰减或放大梯度。

## 3.2 编程：梯度消失/爆炸模拟与初始化策略

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 实验1：Sigmoid + 普通高斯初始化 → 梯度消失
print("=== 实验1: Sigmoid + 普通初始化 (梯度消失) ===")
net1 = nn.Sequential(
    *[nn.Sequential(nn.Linear(256, 256), nn.Sigmoid()) for _ in range(20)]
)
for m in net1.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=1)

X1 = torch.randn(100, 256)
y1 = torch.randint(0, 2, (100,))
loss_fn = nn.CrossEntropyLoss()

# 添加输出层用于分类
net1_with_output = nn.Sequential(net1, nn.Linear(256, 2))
opt = torch.optim.SGD(net1_with_output.parameters(), lr=0.01)

output = net1_with_output(X1)
loss = loss_fn(output, y1)
loss.backward()

grad_norms_sig = []
for i, m in enumerate(net1.modules()):
    if isinstance(m, nn.Linear):
        if m.weight.grad is not None:
            grad_norms_sig.append(m.weight.grad.norm().item())

print(f"前5层梯度范数: {grad_norms_sig[:5]}")
print(f"后5层梯度范数: {grad_norms_sig[-5:]}")
print(f"梯度范围: [{min(grad_norms_sig):.2e}, {max(grad_norms_sig):.2e}]")

In [ ]:
# 实验2：ReLU + 大初始化 → 梯度爆炸风险
print("\n=== 实验2: ReLU + 大方差初始化 (梯度爆炸风险) ===")
net2 = nn.Sequential(
    *[nn.Sequential(nn.Linear(256, 256), nn.ReLU()) for _ in range(20)]
)
for m in net2.modules():
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=10)  # 大方差

net2_with_output = nn.Sequential(net2, nn.Linear(256, 2))

try:
    X2 = torch.randn(100, 256)
    output = net2_with_output(X2)
    loss = loss_fn(output, y1)
    loss.backward()
    
    grad_norms_relu_large = []
    for m in net2.modules():
        if isinstance(m, nn.Linear):
            if m.weight.grad is not None:
                gn = m.weight.grad.norm().item()
                grad_norms_relu_large.append(gn)
                if torch.isnan(loss):
                    print("Loss 变为 NaN - 梯度爆炸")
                    break
    
    print(f"前5层梯度范数: {grad_norms_relu_large[:5]}")
    print(f"后5层梯度范数: {grad_norms_relu_large[-5:]}")
    print(f"梯度范围: [{min(grad_norms_relu_large):.2e}, {max(grad_norms_relu_large):.2e}]")
except RuntimeError as e:
    print(f"发生错误: {e}")

In [ ]:
# 实验3：Xavier初始化 + ReLU → 梯度稳定
print("\n=== 实验3: ReLU + Xavier初始化 (梯度稳定) ===")
net3 = nn.Sequential(
    *[nn.Sequential(nn.Linear(256, 256), nn.ReLU()) for _ in range(20)]
)
for m in net3.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

net3_with_output = nn.Sequential(net3, nn.Linear(256, 2))

X3 = torch.randn(100, 256)
output = net3_with_output(X3)
loss = loss_fn(output, y1)
loss.backward()

grad_norms_xavier = []
for m in net3.modules():
    if isinstance(m, nn.Linear):
        if m.weight.grad is not None:
            grad_norms_xavier.append(m.weight.grad.norm().item())

print(f"前5层梯度范数: {grad_norms_xavier[:5]}")
print(f"后5层梯度范数: {grad_norms_xavier[-5:]}")
print(f"梯度范围: [{min(grad_norms_xavier):.2e}, {max(grad_norms_xavier):.2e}]")

In [ ]:
# 可视化三种方案的梯度分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(range(len(grad_norms_sig)), grad_norms_sig, marker='o', linewidth=2)
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Gradient Norm (log scale)')
axes[0].set_title('Sigmoid + Gauss (梯度消失)')
axes[0].grid()
axes[0].axhline(y=1e-6, color='r', linestyle='--', alpha=0.5, label='消失阈值')
axes[0].legend()

if len(grad_norms_relu_large) > 0:
    axes[1].semilogy(range(len(grad_norms_relu_large)), grad_norms_relu_large, marker='s', linewidth=2)
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Gradient Norm (log scale)')
    axes[1].set_title('ReLU + 大方差 (梯度爆炸风险)')
    axes[1].grid()
    axes[1].axhline(y=1e3, color='r', linestyle='--', alpha=0.5, label='爆炸阈值')
    axes[1].legend()

axes[2].semilogy(range(len(grad_norms_xavier)), grad_norms_xavier, marker='^', linewidth=2, color='green')
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('Gradient Norm (log scale)')
axes[2].set_title('ReLU + Xavier初始化 (稳定)')
axes[2].grid()
axes[2].axhline(y=1e-6, color='r', linestyle='--', alpha=0.3, label='消失阈值')
axes[2].axhline(y=1e3, color='r', linestyle='--', alpha=0.3, label='爆炸阈值')
axes[2].set_ylim([1e-8, 1e4])
axes[2].legend()

plt.tight_layout()
plt.savefig('gradient_stability.png', dpi=100)
plt.show()

print("\n梯度稳定性分析完成")

---
# Part 4: 泛化表现、协变量偏移和对抗性数据

## 4.1 协变量偏移 vs 标签偏移 (理论)

### 协变量偏移 (Covariate Shift)
$$p(x) \neq q(x) \quad \text{but} \quad p(y|x) = q(y|x)$$

**例子：** 医学诊断系统
- 训练集：健康人群（年龄20-40岁）
- 测试集：老年人群（年龄60-80岁）
- 特征分布变化，但给定特征的疾病概率不变

### 标签偏移 (Label Shift)
$$p(y) \neq q(y) \quad \text{but} \quad p(x|y) = q(x|y)$$

**例子：** 垃圾邮件分类
- 训练集：垃圾邮件占10%
- 测试集：垃圾邮件占50%
- 给定类别的特征分布不变，但类别比例变化

### 区别与联系
| 维度 | 协变量偏移 | 标签偏移 |
|------|----------|--------|
| 变化源 | 输入分布 | 标签分布 |
| 影响 | 特征空间 | 决策边界 |
| 解决 | 重权重 | 重加权 |
| 检测 | 密度比 $p(x)/q(x)$ | 边际概率 $p(y)/q(y)$ |

## 4.2 编程：协变量偏移模拟与权重修正

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# 数据构造
np.random.seed(42)

# 训练集 P: N(-1, 1)
X_train = np.random.normal(-1, 1, (1000, 1))
y_train = 2 * X_train.flatten() + np.random.normal(0, 0.1, 1000)  # y = 2x + ε

# 测试集 Q: N(2, 1) - 协变量偏移
X_test = np.random.normal(2, 1, (500, 1))
y_test = 2 * X_test.flatten() + np.random.normal(0, 0.1, 500)

print(f"训练集: X ~ N({X_train.mean():.2f}, {X_train.std():.2f}), size={len(X_train)}")
print(f"测试集: X ~ N({X_test.mean():.2f}, {X_test.std():.2f}), size={len(X_test)}")
print(f"\n注意：标签函数 y=2x+ε 相同，但输入分布明显不同（协变量偏移）")

In [ ]:
# 基线模型：无偏移修正
print("\n=== 基线模型（无修正）===")
model_baseline = LinearRegression()
model_baseline.fit(X_train, y_train)
y_pred_baseline = model_baseline.predict(X_test)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)

print(f"权重: {model_baseline.coef_[0]:.4f}")
print(f"测试MSE: {mse_baseline:.6f}")

In [ ]:
# 偏移校正：逻辑回归估计密度比
print("\n=== 偏移校正（权重修正）===")

# 构造分类问题：0=训练集，1=测试集
X_combined = np.vstack([X_train, X_test])
y_combined = np.hstack([np.zeros(len(X_train)), np.ones(len(X_test))])

# 训练逻辑回归
clf = LogisticRegression()
clf.fit(X_combined, y_combined)

# 预测概率 P(test|x)
probs_test = clf.predict_proba(X_train)[:, 1]  # P(y=1|x) = P(test|x)
probs_train = 1 - probs_test  # P(y=0|x) = P(train|x)

# 计算权重 w_i = P(test|x_i) / P(train|x_i)
weights = probs_test / (probs_train + 1e-8)
weights = weights / weights.mean()  # 归一化

print(f"权重范围: [{weights.min():.4f}, {weights.max():.4f}]")
print(f"权重均值: {weights.mean():.4f}")

In [ ]:
# 加权线性回归
print("\n=== 加权模型（权重修正后）===")
model_weighted = LinearRegression()
model_weighted.fit(X_train, y_train, sample_weight=weights)
y_pred_weighted = model_weighted.predict(X_test)
mse_weighted = mean_squared_error(y_test, y_pred_weighted)

print(f"权重: {model_weighted.coef_[0]:.4f}")
print(f"测试MSE: {mse_weighted:.6f}")
print(f"\n改进: {(mse_baseline - mse_weighted) / mse_baseline * 100:.2f}%")

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 数据分布
ax = axes[0]
ax.hist(X_train, bins=30, alpha=0.6, label='Train P(x)~N(-1,1)', density=True)
ax.hist(X_test, bins=30, alpha=0.6, label='Test Q(x)~N(2,1)', density=True)
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.set_title('协变量偏移：输入分布变化')
ax.legend()
ax.grid(alpha=0.3)

# 模型预测对比
ax = axes[1]
ax.scatter(X_test, y_test, alpha=0.3, s=20, label='Ground Truth', color='black')
ax.scatter(X_test, y_pred_baseline, alpha=0.5, s=20, label=f'Baseline (MSE={mse_baseline:.4f})', color='red')
ax.scatter(X_test, y_pred_weighted, alpha=0.5, s=20, label=f'Weighted (MSE={mse_weighted:.4f})', color='green')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('预测结果对比')
ax.legend()
ax.grid(alpha=0.3)

# 权重分布
ax = axes[2]
ax.hist(weights, bins=30, alpha=0.7, edgecolor='black')
ax.axvline(weights.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean={weights.mean():.2f}')
ax.set_xlabel('Sample Weight w_i')
ax.set_ylabel('Frequency')
ax.set_title('样本权重分布')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('covariate_shift.png', dpi=100)
plt.show()

print("\n协变量偏移修正完成")

---
# 总结

本次作业涵盖的核心内容：

1. **多层感知机 (MLP)**
   - 非线性激活函数的重要性（证明：无激活=单层）
   - Sigmoid和tanh的导数与自身的关系
   - 纯NumPy从零实现单隐藏层MLP

2. **正则化技术**
   - L2权重衰减：直接在梯度更新时衰减权重
   - Dropout：随机丢弃隐元，训练/测试时差异
   - 对比实验：量化不同正则化的效果

3. **数值稳定性**
   - 梯度消失：Sigmoid导数 < 0.25，深层梯度指数衰减
   - 梯度爆炸：权重矩阵特征值 > 1，导致梯度放大
   - 解决方案：Xavier/He初始化 + ReLU激活

4. **分布偏移**
   - 协变量偏移：$p(x) \neq q(x)$ 但条件分布不变
   - 权重修正：用逻辑回归估计密度比，重新加权训练集
   - 验证：加权回归降低测试误差

**关键代码技巧：**
- NumPy实现激活函数、损失函数、反向传播
- 手动实现Dropout（训练/测试时不同行为）
- Xavier初始化：`std = sqrt(1/n_in)`
- 密度比权重：`w = P(test|x) / P(train|x)`